# Коллаборативная фильтрация


**Цель:**  
Обучить и сравнить классические коллаборативные модели:
1. KNN (user based и item based)
3. SVD


**Метрики:** RMSE, MAE  
**Сплит:** Временной (80/20)

## Импорт библиотек

In [14]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

from functools import wraps
import copy
from typing import Dict
import pickle
import os

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold

from surprise import KNNBasic, KNNWithMeans, KNNWithZScore
from surprise import SVD, Dataset, Reader

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

## Загрузка и подготовка данных

### 1. Загрузка данных

In [15]:
# Загрузка
ratings = pd.read_csv('../data/ml-latest-small/ratings.csv')

In [16]:
user_counts = ratings['userId'].value_counts()

In [17]:
user_counts

userId
414    2698
599    2478
474    2108
448    1864
274    1346
       ... 
431      20
442      20
569      20
576      20
595      20
Name: count, Length: 610, dtype: int64

### 2. Предобработка

Для решения проблемы "холодного" старта удалим фильмы, которые оценило малое кол-во пользователей (< 5). В датасете содержатся пользователи с >= 20 проставленными рейтингами, пользователей очищать не будем.

In [18]:
def filter_rare_users_items(ratings, min_item_ratings=5):
    """Удаляет фильмы с малым числом оценок"""
    item_counts = ratings['movieId'].value_counts()
    items_to_keep = item_counts[item_counts >= min_item_ratings].index
    ratings = ratings[ratings['movieId'].isin(items_to_keep)]
    
    return ratings

In [19]:
# Фильтрация
print(f"До фильтрации: {len(ratings)} оценок")
ratings_filtered = filter_rare_users_items(ratings)
print(f"После фильтрации: {len(ratings_filtered)} оценок")

До фильтрации: 100836 оценок
После фильтрации: 90274 оценок


Для разбиения данных будем использовать kfold cross-val.

In [7]:
def cross_validate(n_splits=5, shuffle=True, random_state=42, verbose=True):
    """
    Декоратор для добавления кросс-валидации к методам fit и evaluate
    """
    def decorator(func):
        @wraps(func)
        def wrapper(model, ratings, *args, **kwargs):
            # Если передан готовый train/test сплит, используем его
            if 'train' in kwargs and 'test' in kwargs:
                if verbose:
                    print("Используется готовый train/test сплит")
                return func(model, ratings, *args, **kwargs)
            
            # Иначе выполняем кросс-валидацию
            if verbose:
                print(f"Запуск {n_splits}-fold cross-validation...")
            
            kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
            
            # Получаем данные
            X = ratings[['userId', 'movieId']].values
            y = ratings['rating'].values
            
            fold_results = []
            
            for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
                if verbose:
                    print(f"  Fold {fold}/{n_splits}")
                
                train_data = ratings.iloc[train_idx]
                val_data = ratings.iloc[val_idx]
                
                # Клонируем модель для этого фолда
                fold_model = copy.deepcopy(model)
                
                # Обучаем на train
                fold_model.fit(train_data)
                
                # Вызываем функцию оценки
                fold_result = func(fold_model, ratings, *args, **kwargs)
                fold_results.append(fold_result)
            
            # Агрегируем результаты
            if verbose:
                print(f"\nРезультаты {n_splits}-fold CV:")
            
            if isinstance(fold_results[0], dict):
                aggregated = {}
                for key in fold_results[0].keys():
                    values = [r[key] for r in fold_results]
                    aggregated[f'{key}_mean'] = np.mean(values)
                    if verbose:
                        print(f"  {key}: {aggregated[f'{key}_mean']:.4f}")
                
                # Добавляем все результаты
                aggregated['all_folds'] = fold_results
                return aggregated
            
            return fold_results
        
        return wrapper
    return decorator

Сразу объявим функцию для оценки прогнозов

In [34]:
@cross_validate(n_splits=5, shuffle=True, random_state=42, verbose=False)
def evaluate_predictions(model, test_data, **kwargs):
    """
    Оценка качества предсказаний
    """
    # Если переданы train/test, используем их
    if 'train' in kwargs and 'test' in kwargs:
        train_data = kwargs['train']
        test_data = kwargs['test']
    else:
        train_data = kwargs.get('train_data', ratings)
        test_data = kwargs.get('test_data', ratings)

    predictions = []
    actuals = []
    
    for _, row in test_data.iterrows():
        pred = model.predict(row['userId'], row['movieId'])
        predictions.append(pred)
        actuals.append(row['rating'])
    
    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    mae = mean_absolute_error(actuals, predictions)
    
    return {'RMSE': rmse, 'MAE': mae}

@cross_validate(n_splits=5, shuffle=True, random_state=42, verbose=True)
def evaluate_predictions_verbose(model, test_data, **kwargs):
    """
    Оценка качества предсказаний
    """
    # Если переданы train/test, используем их
    if 'train' in kwargs and 'test' in kwargs:
        train_data = kwargs['train']
        test_data = kwargs['test']
    else:
        train_data = kwargs.get('train_data', ratings)
        test_data = kwargs.get('test_data', ratings)

    predictions = []
    actuals = []
    
    for _, row in test_data.iterrows():
        pred = model.predict(row['userId'], row['movieId'])
        predictions.append(pred)
        actuals.append(row['rating'])
    
    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    mae = mean_absolute_error(actuals, predictions)
    
    return {'RMSE': rmse, 'MAE': mae}

Также создадим функцию для сохранения моделей:

In [44]:
def save_model(model, model_name, metrics=None, params=None):
    os.makedirs('../models', exist_ok=True)
    filename = f"../models/{model_name}.pkl"
    
    model_data = {
        'model': model,
        'metrics': metrics,
        'params': params
    }
    
    with open(filename, 'wb') as f:
        pickle.dump(model_data, f)
    
    print(f"Модель сохранена: {filename}")
    return filename

## Бейзлайн модель - по популярности

In [30]:
class PopularityRecommender:
    """Рекомендует самые популярные фильмы"""
    
    def __init__(self):
        self.popularity = None
        
    def fit(self, ratings):
        self.popularity = ratings.groupby('movieId')['rating'].count().sort_values(ascending=False)
        self.mean_ratings = ratings.groupby('movieId')['rating'].mean()
        
    def recommend(self, n=10):
        return self.popularity.head(n).index.tolist()
    
    def predict(self, user_id, movie_id):
        # Предсказывает оценку (средняя оценка фильма)
        return self.mean_ratings.get(movie_id, 3.5)

In [48]:
pop_model = PopularityRecommender()
metrics_cv = evaluate_predictions(pop_model, ratings_filtered)

In [49]:
metrics_cv

{'RMSE_mean': np.float64(0.9355748249386702),
 'MAE_mean': np.float64(0.72331380383968),
 'all_folds': [{'RMSE': np.float64(0.9355878904370811),
   'MAE': 0.7231809061553649},
  {'RMSE': np.float64(0.9353588609664054), 'MAE': 0.7232013297188628},
  {'RMSE': np.float64(0.9356568468258676), 'MAE': 0.7234422853424796},
  {'RMSE': np.float64(0.9357286421053285), 'MAE': 0.7232068016789585},
  {'RMSE': np.float64(0.9355418843586681), 'MAE': 0.7235376963027347}]}

In [52]:
save_model(
    pop_model,
    'popularity_model',
    metrics=metrics_cv,
    params=None
)

Модель сохранена: ../models/popularity_model.pkl


'../models/popularity_model.pkl'

## KNN

In [43]:
class KNNRecommender:
    """
    Унифицированный класс для KNN-моделей из Surprise
    Поддерживает: KNNBasic, KNNWithMeans, KNNWithZScore
    """
    
    def __init__(self, 
                 model_type='basic',
                 k=40, 
                 min_k=1,
                 sim_name='msd',
                 user_based=True):
        self.k = k
        self.min_k = min_k
        self.model_type = model_type
        self.sim_name = sim_name
        self.user_based = user_based
            
        self.model = None
        
    def fit(self, ratings):
        """Обучение модели"""
        reader = Reader(rating_scale=(0.5, 5.0))
        data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
        trainset = data.build_full_trainset()
        sim_options = {
            'name': self.sim_name,
            'user_based': self.user_based,
            'min_support': 5
        }
        
        if self.model_type == 'basic':
            self.model = KNNBasic(
                k=self.k, 
                min_k=self.min_k,
                sim_options=sim_options,
                verbose=False
            )
        elif self.model_type == 'means':
            self.model = KNNWithMeans(
                k=self.k,
                min_k=self.min_k,
                sim_options=sim_options,
                verbose=False
            )
        elif self.model_type == 'zscore':
            self.model = KNNWithZScore(
                k=self.k,
                min_k=self.min_k,
                sim_options=sim_options,
                verbose=False
            )
        else:
            raise ValueError(f"Unknown model_type: {self.model_type}")
            
        self.model.fit(trainset)
        
    def predict(self, user_id, movie_id):
        """Предсказание оценки"""
        try:
            return self.model.predict(user_id, movie_id).est
        except:
            return 3.5

In [104]:
model = KNNRecommender()
metrics_cv = evaluate_predictions(model, ratings_filtered)

Запуск 5-fold cross-validation...
  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5

Результаты 5-fold CV:
  RMSE: 0.8346
  MAE: 0.6361


### Подбор гиперпарамтеров с Optuna

In [10]:
def objective_knn(trial, ratings, model_type='basic'):
    """
    Целевая функция для оптимизации KNN через Optuna
    """
    model_type = trial.suggest_categorical('model_type', ['basic', 'means', 'zscore'])
    k = trial.suggest_int('k', 5, 100, step=5)
    min_k = trial.suggest_int('min_k', 1, 10)
    sim_name = trial.suggest_categorical('sim_name', ['cosine', 'pearson', 'msd'])
    user_based = trial.suggest_categorical('user_based', [True, False])
    
    model = KNNRecommender(
        model_type=model_type,
        k=k,
        min_k=min_k,
        sim_name=sim_name,
        user_based=user_based
    )
    
    results = evaluate_predictions(model, ratings)
    
    return results['RMSE_mean']

In [11]:
def optimize_knn(ratings, n_trials=100):
    """
    Оптимизация KNN модели через Optuna
    """
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=0), study_name=f'knn')
    
    study.optimize(
        lambda trial: objective_knn(trial, ratings),
        n_trials=n_trials,
        show_progress_bar=True
    )

    print(f"\nЛучшие параметры:")
    for key, value in study.best_params.items():
        print(f"   {key}: {value}")
    print(f"   Лучший RMSE: {study.best_value:.4f}")
    
    return study

In [ ]:
study_knn = optimize_knn(ratings_filtered, n_trials=100)

Лучшие параметры:  
- model_type: means  
- k: 5  
- min_k: 1  
- sim_name: msd  
- user_based: False  
   
Лучший RMSE: 0.6190  

In [61]:
model_params = {
    'model_type': 'means',
    'k': 5,
    'min_k': 1,
    'sim_name': 'msd',
    'user_based': False
}

model = KNNRecommender(**model_params)
metrics_cv = evaluate_predictions_verbose(model, ratings_filtered)

Запуск 5-fold cross-validation...
  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5

Результаты 5-fold CV:
  RMSE: 0.6190
  MAE: 0.4168


In [62]:
save_model(
    model,
    'knn_model',
    metrics=metrics_cv,
    params=model_params
)

Модель сохранена: ../models/knn_model.pkl


'../models/knn_model.pkl'

## SVD

In [47]:
class SVDRecommender:
    def __init__(self, n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02):
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.lr_all = lr_all
        self.reg_all = reg_all
        self.model = None
        
    def fit(self, ratings):
        reader = Reader(rating_scale=(0.5, 5.0))
        data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
        trainset = data.build_full_trainset()
        
        self.model = SVD(
            n_factors=self.n_factors,
            n_epochs=self.n_epochs,
            lr_all=self.lr_all,
            reg_all=self.reg_all,
            random_state=0
        )
        self.model.fit(trainset)
        
    def predict(self, user_id, movie_id):
        try:
            return self.model.predict(user_id, movie_id).est
        except:
            return 3.5

In [26]:
model = SVDRecommender()
metrics_cv = evaluate_predictions(model, ratings_filtered)

Запуск 5-fold cross-validation...
  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5

Результаты 5-fold CV:
  RMSE: 0.7806
  MAE: 0.5964


### Hyperparameters tune

In [28]:
def objective_svd(trial, ratings):
    """Целевая функция для SVD"""
    n_factors = trial.suggest_int('n_factors', 20, 150, step=10)
    n_epochs = trial.suggest_int('n_epochs', 10, 50, step=5)
    lr_all = trial.suggest_float('lr_all', 1e-4, 0.1, log=True)
    reg_all = trial.suggest_float('reg_all', 1e-4, 0.1, log=True)
    
    model = SVDRecommender(
        n_factors=n_factors,
        n_epochs=n_epochs,
        lr_all=lr_all,
        reg_all=reg_all
    )
    
    results = evaluate_predictions(model, ratings)
    return results['RMSE_mean']

In [29]:
def optimize_svd(ratings, n_trials=100):
    """
    Оптимизация SVD модели через Optuna
    """
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=0), study_name=f'svd')
    
    study.optimize(
        lambda trial: objective_svd(trial, ratings),
        n_trials=n_trials,
        show_progress_bar=True
    )

    print(f"\nЛучшие параметры:")
    for key, value in study.best_params.items():
        print(f"   {key}: {value}")
    print(f"   Лучший RMSE: {study.best_value:.4f}")
    
    return study

In [30]:
study_svd = optimize_svd(ratings_filtered, n_trials=100)

  0%|          | 0/100 [00:00<?, ?it/s]


Лучшие параметры:
   n_factors: 130
   n_epochs: 50
   lr_all: 0.052810129457552576
   reg_all: 0.004596241522531441
   Лучший RMSE: 0.5022


Лучшие параметры:
- n_factors: 130  
- n_epochs: 50  
- lr_all: 0.052810129457552576  
- reg_all: 0.004596241522531441

Лучший RMSE: 0.5022

In [64]:
model_params = {
    'n_factors': 130,
    'n_epochs': 50,
    'lr_all': 0.0528,
    'reg_all': 0.0046,
}

model = SVDRecommender(**model_params)
metrics_cv = evaluate_predictions_verbose(model, ratings_filtered)

Запуск 5-fold cross-validation...
  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5

Результаты 5-fold CV:
  RMSE: 0.5022
  MAE: 0.2288


In [65]:
save_model(
    model,
    'svd_model',
    metrics=metrics_cv,
    params=model_params
)

Модель сохранена: ../models/svd_model.pkl


'../models/svd_model.pkl'

## Оценка для сравнения с другими моделями

In [8]:
# check data
print(len(ratings_filtered))

90274


Разделим данные на train/test в пропорциях 80%/20%. Также учтём timestamp:

In [1]:
def time_train_test_split(df, train_size=0.8):
    df.drop_duplicates(subset=['userId', 'movieId', 'timestamp'], inplace=True)
    
    df_sorted = df.sort_values(by='timestamp')
    df_sorted['item_id'] = df_sorted['movieId']
    grouped = df_sorted.groupby('item_id')

    def split_train_test(group):
        train_size_ = int(train_size * len(group))
        return group.iloc[:train_size_], group.iloc[train_size_:]

    train, test = zip(*grouped.apply(split_train_test))

    train = pd.concat(train)
    test = pd.concat(test)

    return train, test

In [23]:
train_df, test_df = time_train_test_split(ratings_filtered)

print('Размеры выборок:')
print(f'train: {len(train_df)}')
print(f'test: {len(test_df)}')

Размеры выборок:
train: 70752
test: 19522


In [39]:
def evaluate_classical_model(model, train_df, test_df, k=50, threshold=4.0):
    """
    Оценка классических моделей по 4 метрикам:
    - RMSE
    - MAE
    - Recall@K
    - NDCG@K
    """
    
    # 1. RMSE и MAE 
    predictions = []
    actuals = []
    
    for _, row in test_df.iterrows():
        pred = model.predict(row['userId'], row['movieId'])
        predictions.append(pred)
        actuals.append(row['rating'])
    
    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    mae = mean_absolute_error(actuals, predictions)
    
    # 2. Recall@K и NDCG@K
    recall_list = []
    ndcg_list = []
    candidates = pd.concat([train_df['movieId'], test_df['movieId']]).unique()
    movie_to_idx = {mid: i for i, mid in enumerate(candidates)}
    
    for user_id in test_df['userId'].unique():
        
        # Релевантные фильмы из test (оценки >= threshold)
        relevant_items = test_df[
            (test_df['userId'] == user_id) & 
            (test_df['rating'] >= threshold)
        ]['movieId'].values
        
        if len(relevant_items) == 0:
            continue
        
        # Делаем предсказания для всех кандидатов
        predictions_all = []
        for movie_id in candidates:
            pred = model.predict(user_id, movie_id)
            predictions_all.append((movie_to_idx[movie_id], pred))
        
        # Сортируем по убыванию предсказания
        predictions_all.sort(key=lambda x: x[1], reverse=True)
        top_k = [item_idx for item_idx, _ in predictions_all[:k]]
        
        relevant_set = set(movie_to_idx[m] for m in relevant_items)
        recommended_set = set(top_k)
        
        # Recall@K
        recall = len(relevant_set & recommended_set) / len(relevant_set)
        recall_list.append(recall)
        
        # NDCG@K
        dcg = 0
        for i, item_idx in enumerate(top_k):
            if item_idx in relevant_set:
                dcg += 1 / np.log2(i + 2)
        
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(relevant_items), k)))
        ndcg = dcg / idcg if idcg > 0 else 0
        ndcg_list.append(ndcg)
    
    return {
        'RMSE': rmse,
        'MAE': mae,
        f'Recall@{k}': np.mean(recall_list) if recall_list else 0,
        f'NDCG@{k}': np.mean(ndcg_list) if ndcg_list else 0
    }

In [40]:
def fit_eval(model, train_df, test_df):
    model.fit(train_df)
    return evaluate_classical_model(model, train_df, test_df)

### Basic popularity

In [41]:
pop_model = PopularityRecommender()
metrics = fit_eval(pop_model, train_df, test_df)

for key in metrics.keys():
    print(f'{key}: {metrics[key]:.4f}')

RMSE: 0.9918
MAE: 0.7627
Recall@50: 0.0175
NDCG@50: 0.0095


### KNN

In [45]:
model_params = {
    'model_type': 'means',
    'k': 5,
    'min_k': 1,
    'sim_name': 'msd',
    'user_based': False
}
knn_model = KNNRecommender(**model_params)
metrics = fit_eval(knn_model, train_df, test_df)

for key in metrics.keys():
    print(f'{key}: {metrics[key]:.4f}')

RMSE: 0.9539
MAE: 0.7267
Recall@50: 0.0287
NDCG@50: 0.0217


### SVD

In [48]:
model_params = {
    'n_factors': 130,
    'n_epochs': 50,
    'lr_all': 0.0528,
    'reg_all': 0.0046,
}
svd_model = SVDRecommender(**model_params)
metrics = fit_eval(svd_model, train_df, test_df)

for key in metrics.keys():
    print(f'{key}: {metrics[key]:.4f}')

RMSE: 0.9022
MAE: 0.6922
Recall@50: 0.0518
NDCG@50: 0.0470
